# 1.2 — Inventory review

Applies `INVENTORY_REVIEW` from [ai_lca_config.py](ai_lca_config.py) on top of
the flow list from notebook 1.1: exclude a flow, or edit its amount/unit/
direction/notes/name. Only flows listed in `INVENTORY_REVIEW` change.

Source evidence columns (`document`, `page`, `evidence_text`, ...) are never
edited here — they stay attached to the original AI-extracted values for the
audit trail, whatever else changes on that row.

In [ ]:
import pandas as pd

import ai_lca_config as cfg
cfg.print_config()

from ai_lca.export import normalize_inventory_review
from ai_lca.notebook_helpers import inventory_review_dataframe, load_extraction, run_output_dir

run_dir = run_output_dir(cfg.OUTPUT_DIR, cfg.RUN_LABEL)
raw_path = run_dir / "1_extraction_raw.json"
reviewed_path = run_dir / "1_1_extraction_reviewed.json"
inventory_path = run_dir / "1_1_inventory_after_process_review.csv"
for p in (raw_path, reviewed_path, inventory_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found — run 1.paper_ingest_and_extract.ipynb and "
                                 "1.1.paper_process_review.ipynb first.")

original_extraction = load_extraction(raw_path)
reviewed_extraction = load_extraction(reviewed_path)
inventory_df = pd.read_csv(inventory_path)
print(f"Loaded {len(inventory_df)} flow(s) from {inventory_path}")
print()
print("Inventory BEFORE review:")
inventory_df

## Apply `INVENTORY_REVIEW`

In [ ]:
edited_df = inventory_review_dataframe(inventory_df, cfg.INVENTORY_REVIEW)
edited_df = normalize_inventory_review(
    edited_df, extraction=reviewed_extraction, original_extraction=original_extraction,
)

status_counts = edited_df["review_status"].value_counts().to_dict()
included = int(edited_df["include"].sum())
print(f"{included} of {len(edited_df)} flow(s) included.")
print("Review status:", status_counts)

### Inventory AFTER review

In [ ]:
edited_df

## Save reviewed inventory

In [ ]:
out_path = run_dir / "1_2_inventory_reviewed.csv"
edited_df.to_csv(out_path, index=False)
print("Saved reviewed inventory to:", out_path)
print()
by_direction = edited_df[edited_df["include"]].groupby("direction").size().to_dict()
print("Included flows by direction:", by_direction)
print()
print("Next: run 1.3.paper_brightway_matching.ipynb.")